In [5]:
# ============================================================
# Q1 - Finding solutions of linear systems
#
# This program:
# 1. Constructs the augmented matrix [A | b]
# 2. Finds REF and RREF using Gaussian elimination
# 3. Identifies pivot and non-pivot columns
# 4. Finds a particular solution
# 5. Finds solutions of Ax = 0
# 6. Finds and verifies the general solution
#
# No built-in functions such as solve(), matrix_rank()
# or rref() are used.
# ============================================================

import numpy as np


# Fixed seed so that the same random values are generated
np.random.seed(42)

np.set_printoptions(
    precision=8,
    suppress=True,
    linewidth=140
)

# Tolerance used for checking zero values
TOL = 1e-9


# ------------------------------------------------------------
# Function to convert a matrix into Row Echelon Form (REF)
# ------------------------------------------------------------

def to_ref(M):
    """
    Performs Gaussian elimination and returns REF.
    Partial pivoting is used while selecting pivots.
    """

    M = M.astype(float).copy()

    rows, cols = M.shape

    pivot_row = 0
    pivot_cols = []

    # Check columns from left to right
    for col in range(cols):

        if pivot_row >= rows:
            break

        # Select the row with the largest absolute value
        # in the current column
        candidate = (
            np.argmax(
                np.abs(M[pivot_row:, col])
            )
            + pivot_row
        )

        # Division by zero handling:
        # If the candidate is approximately zero, there is
        # no pivot in this column, so move to the next column.
        if abs(M[candidate, col]) < TOL:
            continue

        # Move the selected row to the pivot position
        M[[pivot_row, candidate]] = (
            M[[candidate, pivot_row]]
        )

        # Make all entries below the pivot zero
        for r in range(pivot_row + 1, rows):

            factor = (
                M[r, col]
                / M[pivot_row, col]
            )

            M[r, :] = (
                M[r, :]
                - factor * M[pivot_row, :]
            )

        pivot_cols.append(col)
        pivot_row += 1

    return M, pivot_cols


# ------------------------------------------------------------
# Function to convert a matrix into Reduced Row Echelon
# Form (RREF)
# ------------------------------------------------------------

def to_rref(M):
    """
    Converts a matrix to RREF using the REF obtained
    from Gaussian elimination.
    """

    R, pivot_cols = to_ref(M)

    # Start from the last pivot and move upwards
    for i in reversed(range(len(pivot_cols))):

        col = pivot_cols[i]
        row = i

        # Make the pivot equal to 1
        R[row, :] = (
            R[row, :]
            / R[row, col]
        )

        # Make all entries above the pivot zero
        for r in range(row):

            R[r, :] = (
                R[r, :]
                - R[r, col] * R[row, :]
            )

    return R, pivot_cols


# ------------------------------------------------------------
# Find pivot columns, non-pivot columns, particular solution
# and solutions of Ax = 0
# ------------------------------------------------------------

def analyse_system(A, b):
    """
    Finds the REF, RREF, pivot columns, free columns,
    particular solution and null-space basis.
    """

    m, n = A.shape

    # Construct augmented matrix [A | b]
    aug = np.hstack([
        A,
        b.reshape(-1, 1)
    ])

    # Find REF and RREF
    REF, _ = to_ref(aug)
    RREF, piv_all = to_rref(aug)

    # If the last column is a pivot column, then the system
    # contains an equation of the form 0 = non-zero.
    consistent = not any(
        c == n for c in piv_all
    )

    # Pivot columns belonging to A
    pivot_cols = [
        c for c in piv_all
        if c < n
    ]

    # Columns without pivots are free columns
    free_cols = [
        c for c in range(n)
        if c not in pivot_cols
    ]

    # --------------------------------------------------------
    # Particular solution
    #
    # Set all free variables equal to zero.
    # --------------------------------------------------------

    x_p = np.zeros(n)

    for i, c in enumerate(pivot_cols):
        x_p[c] = RREF[i, n]

    # --------------------------------------------------------
    # Solutions of Ax = 0
    #
    # One null-space vector is obtained for every free
    # variable.
    # --------------------------------------------------------

    null_basis = []

    for f in free_cols:

        v = np.zeros(n)

        # Set the selected free variable equal to 1
        v[f] = 1.0

        # Calculate the corresponding pivot variables
        for i, c in enumerate(pivot_cols):
            v[c] = -RREF[i, f]

        null_basis.append(v)

    return (
        REF,
        RREF,
        pivot_cols,
        free_cols,
        x_p,
        null_basis,
        consistent
    )


# ------------------------------------------------------------
# Q1(3)
# Random 5 x 7 matrix A and vector b
# ------------------------------------------------------------

A = np.random.randn(5, 7)

b = np.random.randn(5)


print("Q1(3) Random system")

print("\nA =")
print(A)

print("\nb =")
print(b)


# ------------------------------------------------------------
# Construct augmented matrix and perform REF/RREF
# ------------------------------------------------------------

(
    REF,
    RREF,
    pivot_cols,
    free_cols,
    x_p,
    null_basis,
    consistent
) = analyse_system(A, b)


print("\nAugmented matrix [A | b] =")
print(
    np.hstack([
        A,
        b.reshape(-1, 1)
    ])
)


print("\nREF of [A | b] =")
print(REF)


print("\nRREF of [A | b] =")
print(RREF)


# ------------------------------------------------------------
# Pivot and non-pivot columns
# ------------------------------------------------------------

print("\nConsistent system =", consistent)

print("Pivot columns     =", pivot_cols)

print("Non-pivot columns =", free_cols)


# ------------------------------------------------------------
# Particular solution
# ------------------------------------------------------------

print("\nParticular solution x_p =")
print(x_p)

print("\nVerification of A x_p = b")
print("A @ x_p =", A @ x_p)
print("b       =", b)


# ------------------------------------------------------------
# Solutions of Ax = 0
# ------------------------------------------------------------

print("\nSolutions of Ax = 0:")

for j, v in enumerate(null_basis, 1):

    print(
        f"n{j} =",
        v
    )

    print(
        f"A @ n{j} =",
        A @ v
    )


# ------------------------------------------------------------
# General solution
#
# x = x_p + c1*n1 + c2*n2 + ...
# ------------------------------------------------------------

coeffs = np.random.randn(
    len(null_basis)
)


x_general = (
    x_p
    + sum(
        c * v
        for c, v in zip(
            coeffs,
            null_basis
        )
    )
)


print("\nRandom values of free variables:")
print(coeffs)


print("\nGeneral solution:")
print("x = x_p + c1*n1 + c2*n2 + ...")

print("\nOne general solution obtained using random coefficients:")
print(x_general)


# ------------------------------------------------------------
# Verification of the general solution
# ------------------------------------------------------------

print("\nVerification of general solution")

print("A @ x_general =")
print(A @ x_general)

print("\nb =")
print(b)

print(
    "\nMaximum absolute error =",
    np.max(
        np.abs(
            A @ x_general - b
        )
    )
)

Q1(3) Random system

A =
[[ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337 -0.23413696  1.57921282]
 [ 0.76743473 -0.46947439  0.54256004 -0.46341769 -0.46572975  0.24196227 -1.91328024]
 [-1.72491783 -0.56228753 -1.01283112  0.31424733 -0.90802408 -1.4123037   1.46564877]
 [-0.2257763   0.0675282  -1.42474819 -0.54438272  0.11092259 -1.15099358  0.37569802]
 [-0.60063869 -0.29169375 -0.60170661  1.85227818 -0.01349722 -1.05771093  0.82254491]]

b =
[-1.22084365  0.2088636  -1.95967012 -1.32818605  0.19686124]

Augmented matrix [A | b] =
[[ 0.49671415 -0.1382643   0.64768854  1.52302986 -0.23415337 -0.23413696  1.57921282 -1.22084365]
 [ 0.76743473 -0.46947439  0.54256004 -0.46341769 -0.46572975  0.24196227 -1.91328024  0.2088636 ]
 [-1.72491783 -0.56228753 -1.01283112  0.31424733 -0.90802408 -1.4123037   1.46564877 -1.95967012]
 [-0.2257763   0.0675282  -1.42474819 -0.54438272  0.11092259 -1.15099358  0.37569802 -1.32818605]
 [-0.60063869 -0.29169375 -0.60170661  1.8522781

In [6]:
# ============================================================
# Q2 - Dataset, Rank, Covariance and Power Method
#
# The dataset has 500 rows and 6 features.
#
# f1, f2, f3 and f4 are random standard normal features.
# f5 = 2f1 + 3f2
# f6 = f3 - 2f4
#
# Rank and Power Method are implemented manually.
# np.linalg.eigh() is used only in part (d).
# ============================================================

import numpy as np


# Fixing the seed so that results can be reproduced
np.random.seed(42)

np.set_printoptions(
    precision=8,
    suppress=True,
    linewidth=140
)

TOL = 1e-8


# ------------------------------------------------------------
# Calculate vector norm
# ------------------------------------------------------------

def vnorm(x):
    """Returns the Euclidean norm of vector x."""
    return (x @ x) ** 0.5


# ------------------------------------------------------------
# Q2(1)
# Generate the dataset X
# ------------------------------------------------------------

n = 500


# Four random features from standard normal distribution
f1 = np.random.randn(n)
f2 = np.random.randn(n)
f3 = np.random.randn(n)
f4 = np.random.randn(n)


# Two dependent features
f5 = 2 * f1 + 3 * f2
f6 = f3 - 2 * f4


# Create the dataset
X = np.column_stack([
    f1,
    f2,
    f3,
    f4,
    f5,
    f6
])


print("Q2(1) Dataset X")

print("\nShape of X =", X.shape)

print("\nFirst 10 rows of X:")
print(X[:10])


# ------------------------------------------------------------
# Q2(2)
# Find rank of X using Gaussian elimination
# ------------------------------------------------------------

def rank_via_elimination(M):
    """
    Finds the rank by counting the number of pivots
    obtained during Gaussian elimination.
    """

    M = M.astype(float).copy()

    rows, cols = M.shape

    pivot_row = 0
    rank = 0

    # Process each column
    for col in range(cols):

        if pivot_row >= rows:
            break

        # Find the largest absolute value below the
        # current pivot row
        candidate = (
            np.argmax(
                np.abs(M[pivot_row:, col])
            )
            + pivot_row
        )

        # No pivot in this column
        if abs(M[candidate, col]) < TOL:
            continue

        # Swap rows
        M[[pivot_row, candidate]] = (
            M[[candidate, pivot_row]]
        )

        # Eliminate values below pivot
        for r in range(pivot_row + 1, rows):

            M[r, :] -= (
                M[r, col]
                / M[pivot_row, col]
            ) * M[pivot_row, :]

        rank += 1
        pivot_row += 1

    return rank


print("\nQ2(2) Rank of X")

print(
    "rank(X) =",
    rank_via_elimination(X)
)

print(
    "Expected rank = 4 because f5 and f6 "
    "are linear combinations of the first four features."
)


# ------------------------------------------------------------
# Q2(3a)
# Covariance matrix
#
# C = (1/n) X^T X
# ------------------------------------------------------------

C = (X.T @ X) / n


print("\nQ2(3a) Covariance matrix")

print("Shape of C =", C.shape)

print("\nC =")
print(C)


# ------------------------------------------------------------
# Q2(3b)
# Power Method for dominant eigenvalue
# ------------------------------------------------------------

def power_method(
    M,
    max_iter=100000,
    tol=1e-12,
    seed=0
):
    """
    Finds the dominant eigenvalue and eigenvector
    using the Power Method.
    """

    rng = np.random.default_rng(seed)

    # Start with a random vector
    v = rng.standard_normal(
        M.shape[0]
    )

    # Make the vector unit length
    v = v / vnorm(v)

    lam_old = 0.0

    history = []

    for k in range(
        1,
        max_iter + 1
    ):

        # Multiply matrix by current vector
        w = M @ v

        # Normalise the result
        v = w / vnorm(w)

        # Rayleigh quotient
        lam = v @ (M @ v)

        history.append(lam)

        # Check convergence
        if abs(lam - lam_old) < tol:
            return (
                lam,
                v,
                k,
                history
            )

        lam_old = lam

    return (
        lam,
        v,
        max_iter,
        history
    )


# Find largest eigenvalue and corresponding eigenvector
lam1, v1, it1, _ = power_method(
    C,
    seed=1
)


print("\nQ2(3b) Power Method")

print("\nlambda_1 =", lam1)

print("\nv_1 =")
print(v1)

print(
    "\nNumber of iterations =",
    it1
)


# ------------------------------------------------------------
# Q2(3c)
# Power Method with deflation
#
# After finding an eigenvector, its direction is removed
# from the matrix before finding the next eigenpair.
#
# M = C - (sum vj vj^T) C
# ------------------------------------------------------------

def power_method_deflation(
    C,
    k,
    seed=1
):
    """
    Finds k eigenpairs using the Power Method
    with deflation.
    """

    eigvals = []
    eigvecs = []

    for i in range(k):

        # Start with C for every new eigenvalue
        M = C.copy()

        # Remove the directions already found
        for vj in eigvecs:

            M = (
                M
                - np.outer(vj, vj) @ M
            )

        # Apply Power Method
        lam, v, _, _ = power_method(
            M,
            seed=seed + i
        )

        eigvals.append(lam)
        eigvecs.append(v)

    return (
        np.array(eigvals),
        np.array(eigvecs)
    )


# Rank is 4, so we find 4 non-zero eigenvalues
k = 4


pm_vals, pm_vecs = power_method_deflation(
    C,
    k
)


print(
    "\nQ2(3c) Power Method with Deflation"
)


for i in range(k):

    print(
        f"\nlambda_{i + 1} = "
        f"{pm_vals[i]:.8f}"
    )

    print(
        f"v_{i + 1} ="
    )

    print(pm_vecs[i])


# ------------------------------------------------------------
# Q2(3d)
# Use Python library function to find eigenvalues
# and eigenvectors.
#
# np.linalg.eigh() is allowed in this part.
# ------------------------------------------------------------

w, V = np.linalg.eigh(C)


# Arrange eigenvalues from largest to smallest
order = np.argsort(w)[::-1]

w = w[order]
V = V[:, order]


print(
    "\nQ2(3d) Eigenvalues and eigenvectors "
    "using np.linalg.eigh()"
)


print("\nAll eigenvalues:")
print(w)


for i in range(k):

    print(
        f"\nlambda_{i + 1} (library) = "
        f"{w[i]:.8f}"
    )

    print(
        f"v_{i + 1} (library) ="
    )

    print(V[:, i])


# ------------------------------------------------------------
# Compare Power Method results with library results
# ------------------------------------------------------------

print(
    "\nComparison of Power Method and "
    "library results:"
)


for i in range(k):

    # Eigenvectors can differ only by their sign
    same_dir = np.sign(
        pm_vecs[i] @ V[:, i]
    )

    eig_diff = abs(
        pm_vals[i] - w[i]
    )

    vec_diff = vnorm(
        pm_vecs[i]
        - same_dir * V[:, i]
    )

    print(
        f"\nlambda_{i + 1}: "
        f"PM = {pm_vals[i]:.8f}, "
        f"Library = {w[i]:.8f}"
    )

    print(
        f"Eigenvalue difference = "
        f"{eig_diff:.2e}"
    )

    print(
        f"Eigenvector difference = "
        f"{vec_diff:.2e}"
    )


# ------------------------------------------------------------
# Q2(3e)
# Number of iterations required to reach
# accuracy of 10^-7
# ------------------------------------------------------------

def iters_to_accuracy(
    C,
    eigvecs_found,
    true_lambda,
    target=1e-7,
    seed=1,
    idx=0
):
    """
    Finds the number of Power Method iterations
    required to reach the specified accuracy.
    """

    M = C.copy()

    # Deflate eigenvectors already obtained
    for vj in eigvecs_found:

        M = (
            M
            - np.outer(vj, vj) @ M
        )

    # Run Power Method and store its history
    _, _, _, hist = power_method(
        M,
        seed=seed + idx
    )

    # Check when the required accuracy is reached
    for k_iter, lam in enumerate(
        hist,
        1
    ):

        if abs(
            lam - true_lambda
        ) < target:

            return k_iter

    return len(hist)


print(
    "\nQ2(3e) Iterations required "
    "for accuracy 10^-7"
)


# Store eigenvectors found so far
found = []


for i in range(k):

    it = iters_to_accuracy(
        C,
        found,
        w[i],
        idx=i
    )

    # Ratio of consecutive eigenvalues
    if i + 1 < len(w):

        gap_ratio = abs(
            w[i + 1] / w[i]
        )

    else:

        gap_ratio = 0

    print(
        f"\nlambda_{i + 1}: "
        f"{it} iterations"
    )

    print(
        f"Eigenvalue gap ratio = "
        f"{gap_ratio:.4f}"
    )

    # Add current eigenvector for next deflation
    found.append(
        pm_vecs[i]
    )

Q2(1) Dataset X

Shape of X = (500, 6)

First 10 rows of X:
[[ 0.49671415  0.92617755  1.39935544  0.77836108  3.77196095 -0.15736672]
 [-0.1382643   1.90941664  0.92463368 -0.55118572  5.45172132  2.02700512]
 [ 0.64768854 -1.39856757  0.05963037 -0.81819888 -2.90032565  1.69602814]
 [ 1.52302986  0.56296924 -0.64693678 -0.00337446  4.73496742 -0.64018786]
 [-0.23415337 -0.65064257  0.69822331 -0.17018462 -2.42023446  1.03859256]
 [-0.23413696 -0.48712538  0.39348539 -0.45322805 -1.92965007  1.29994148]
 [ 1.57921282 -0.59239392  0.89519322  0.69638745  1.38124386 -0.49758167]
 [ 0.76743473 -0.86399077  0.6351718   0.95530521 -1.05710285 -1.27543862]
 [-0.46947439  0.04852163  1.04955272  0.08840689 -0.79338389  0.87273894]
 [ 0.54256004 -0.83095012 -0.53523521  1.47753008 -1.40773026 -3.49029537]]

Q2(2) Rank of X
rank(X) = 4
Expected rank = 4 because f5 and f6 are linear combinations of the first four features.

Q2(3a) Covariance matrix
Shape of C = (6, 6)

C =
[[ 0.96097898 -0.0722